# 07. Pandas: analiza

Czas: ok. 30 min

**Czego się nauczysz**
- policzyć nową kolumnę z działania na dwóch kolumnach (cena za Mg) i sprawdzić, czy wynik ma sens,
- policzyć odsetki: ile przetargów ma 1, 2, 3 oferty, ogółem i per rok,
- `groupby`, czyli tabela przestawna: średnia, mediana, kwartyle i IQR ceny per rok, trendy rok do roku,
- korelacje: czy liczba ofert idzie w parze z ceną, wartością, wolumenem i okresem umowy,
- zapisać gotowe tabele do Excela i narysować prosty wykres do raportu.

## 0. Wczytanie czystego pliku i założenia

Pracujemy na czystym pliku, czyli wyniku notebooka 06 (Pandas: czyszczenie danych). Excel nie zapamiętał typu `Int64` (liczba całkowita z pustymi polami), więc `liczba_ofert` wraca jako `1.0`, `2.0`; przywracamy go jedną linią.
Na zajęciach sam przepisujesz kod z sekcji 1 (Nowa kolumna: cena za Mg) i 2 (Odsetek przetargów wg liczby ofert), bo to serce analizy: nowa kolumna, sprawdzenie sensu, odsetki ofert; uzupełniasz też obie komórki **Twoja kolej**: o dwóch ofertach na końcu sekcji 2 (Odsetek przetargów wg liczby ofert) i o medianie na początku sekcji 3 (Cena za Mg per rok: średnia, mediana, kwartyle, IQR). Resztę tej sekcji oraz sekcje 4 (Trendy rok do roku), 5 (Korelacje) i 6 (Eksport wyników do Excela) uruchamiasz i czytasz, a wracasz do nich w domu. Sekcje 7 (Wykres) i 8 (Koncentracja wykonawców) są opcjonalne, jeśli zostanie czas.

In [ ]:
import pandas as pd   # pandas = biblioteka do tabel; "pd" to przyjęty skrót

# wczytujemy CZYSTY plik, wynik notebooka 06 (Pandas: czyszczenie danych);
# gdybyś nie zbudował swojego, gotowy jest w folderze data
df = pd.read_excel("data/przetargi_clean.xlsx")
# astype("Int64") = z powrotem liczba całkowita, która toleruje puste pola (Excel tego typu nie zapisuje)
df["liczba_ofert"] = df["liczba_ofert"].astype("Int64")
# puste pole w tej kolumnie wyświetla się jako <NA>, w kolumnach z przecinkiem jako NaN;
# jedno i drugie znaczy to samo: brak danych
print("Wiersze:", len(df), "| typ liczba_ofert:", df["liczba_ofert"].dtype)
df.head()

**ZAŁOŻENIA do potwierdzenia** (przyjęte przez trenera, do sprawdzenia z Tobą przed raportem):
1. Rok analizy = rok ogłoszenia przetargu (kolumna `rok` z `data_ogloszenia`).
2. `wartosc_pln` to łączna wartość umowy brutto za cały okres, `wolumen_mg` to łączny wolumen za cały okres, więc `cena_za_mg = wartosc_pln / wolumen_mg` bez przeliczania na rok.
3. Cena za Mg powyżej 3000 zł to błąd danych (źle zescrapowany wolumen): zamieniamy ją na pustą, nie usuwamy wiersza.
4. Kody odpadów analizujemy per przetarg, nie per kod (rozbicie kodów na wiersze to praca domowa).
5. Do analiz liczby ofert bierzemy tylko wiersze ze znaną liczbą ofert.
6. W sekcji 8 (Koncentracja wykonawców) "rynek" = wszystkie przetargi w zbiorze, udziały wg sumy wartości wygranych umów, każda `MZK <Gmina>` to osobna firma.

## 1. Nowa kolumna: cena za Mg

Działanie na dwóch kolumnach pandas wykonuje wiersz po wierszu, jak formuła przeciągnięta w dół całego arkusza; tam, gdzie brakuje wartości albo wolumenu, wynik jest pusty (`NaN`), bez błędu.
`describe()` liczy statystyki opisowe: `count` = ile wypełnionych, `mean` = średnia, `std` = odchylenie standardowe (typowy rozrzut wokół średniej), `min`, `max`, `50%` = mediana; `25%` i `75%` to kwartyle: cena, poniżej której jest 25% (odpowiednio 75%) przetargów.

In [ ]:
# kolumna dzielona przez kolumnę: pandas liczy dla każdego wiersza osobno (założenie 2)
df["cena_za_mg"] = df["wartosc_pln"] / df["wolumen_mg"]
# describe() = statystyki opisowe nowej kolumny; round(1) zaokrągla do 1 miejsca po przecinku
df["cena_za_mg"].describe().round(1)

Typowa cena to 400–1200 zł/Mg, a `max` to ponad 11 000 zł. Zanim policzymy cokolwiek dalej, sprawdzamy sensowność: patrzymy na najdroższe przetargi.

In [ ]:
# sort_values(..., ascending=False) = od najdroższej; wynik zapisujemy do zmiennej
top_prices = df.sort_values("cena_za_mg", ascending=False)
# podwójne nawiasy = lista kolumn do pokazania; head(8) = 8 pierwszych wierszy;
# round(1) = czytelne kwoty zamiast zapisu typu 5.6e+07 (5,6 razy 10 do potęgi 7, czyli 56 mln)
top_prices[["gmina", "rok", "wartosc_pln", "wolumen_mg", "cena_za_mg"]].head(8).round(1)

Pięć przetargów ma wolumen ok. 10 razy za mały (błąd scrapowania), stąd absurdalna cena. Zgodnie z założeniem 3 czyścimy samą cenę, wiersz zostaje, bo liczba ofert i wartość są w porządku.

In [ ]:
# df.loc[mask, "kolumna"] = wartość: wpisuje wartość TYLKO w wierszach, gdzie mask ma True;
# mask = warunek na całej kolumnie (tu: cena > 3000), czyli kolumna True/False;
# None = puste pole, pandas zamieni je na NaN
df.loc[df["cena_za_mg"] > 3000, "cena_za_mg"] = None
# Pułapka: df[mask]["cena_za_mg"] = None (dwa nawiasy po kolei) NIE zapisze zmiany;
# zawsze df.loc[mask, "kolumna"]
df["cena_za_mg"].describe().round(1)

Średnia kontra mediana: przed czyszczeniem średnia wynosiła 830 zł, a mediana 751; po czyszczeniu 751 i 749. Pięć błędnych wierszy na 439 przesunęło średnią o 80 zł, mediany prawie nie ruszyło.
Dlatego w raporcie o cenach bezpieczniej opierać się na medianie (lub podawać obie).

## 2. Odsetek przetargów wg liczby ofert

Zgodnie z założeniem 5 bierzemy tylko wiersze ze znaną liczbą ofert: `dropna(subset=[...])` usuwa wiersze z pustym polem w tej kolumnie.
`value_counts(normalize=True)` daje udziały (sumują się do 1) zamiast liczb.

In [ ]:
# dropna(subset=[...]) = usuń wiersze, gdzie liczba_ofert jest pusta; copy() = osobna kopia tabeli
# Pułapka: bez copy() starszy pandas ostrzega (SettingWithCopyWarning),
# gdy do przefiltrowanej tabeli dopisujemy kolumnę
offers = df.dropna(subset=["liczba_ofert"]).copy()
print("Przetargów ze znaną liczbą ofert:", len(offers), "z", len(df))
# value_counts(normalize=True) = udziały zamiast liczb; sort_index() układa wg indeksu,
# czyli etykiet wierszy z lewej (tu: liczba ofert 1, 2, 3...)
offers_share = offers["liczba_ofert"].value_counts(normalize=True).sort_index()
# razy 100 i round(1), żeby czytać jak procenty; rename() = czytelna nazwa kolumny do Excela
offers_share = (offers_share * 100).round(1).rename("udzial_proc")
offers_share

To samo per rok robi `pd.crosstab`, czyli tabela przestawna: wiersze = rok, kolumny = liczba ofert. `normalize="index"` sprawia, że każdy wiersz (rok) sumuje się do 100%.

In [ ]:
# crosstab(wiersze, kolumny) = tabela przestawna, która zlicza przetargi w każdej parze rok x liczba ofert;
# normalize="index" = procenty liczone w obrębie każdego roku
offers_by_year = pd.crosstab(offers["rok"], offers["liczba_ofert"], normalize="index")
# razy 100 i round(1), żeby czytać jak procenty
offers_by_year = (offers_by_year * 100).round(1)
offers_by_year

Najprostsza miara konkurencji do raportu: udział przetargów z jedną ofertą. Porównanie `== 1` na całej kolumnie daje kolumnę `True`/`False` (tak jak porównanie dwóch liczb daje `True` albo `False`), a średnia z takiej kolumny to udział `True`, bo `True` liczy się jak 1, a `False` jak 0.

In [ ]:
# nowa kolumna True/False: czy była dokładnie jedna oferta;
# == to porównanie (pytanie), = to przypisanie (wpisanie do zmiennej); łatwo je pomylić
offers["jedna_oferta"] = offers["liczba_ofert"] == 1
# mean() z True/False = udział True;
# w czystym Pythonie ten udział wymagałby pętli po wszystkich wierszach; w pandas to jedna linia
single_bid_share_total = offers["jedna_oferta"].mean()
# f-string = tekst z literą f przed cudzysłowem, {nawiasy klamrowe} wstawiają wartość;
# :.1% pokazuje ułamek jako procent z 1 miejscem po przecinku
print(f"Udział przetargów z jedną ofertą ogółem: {single_bid_share_total:.1%}")
# groupby("rok") = "dla każdego roku osobno" (tabela przestawna); potem ta sama średnia
# w każdej grupie, w procentach. Rok staje się indeksem (pogrubioną etykietą wiersza z lewej,
# nie kolumną), więc wartość dla 2024 to single_bid_by_year[2024]
single_bid_by_year = (offers.groupby("rok")["jedna_oferta"].mean() * 100).round(1)
single_bid_by_year

Udział jednej oferty rośnie z ok. 41% (2019) do ok. 68% (2024): konkurencja w przetargach w całym okresie wyraźnie słabnie.

**Twoja kolej:** ten sam schemat dla dwóch ofert.

In [ ]:
# TODO: zmień 1 na 2 w linii poniżej i uruchom; zanim to zrobisz, wynik
# jest (błędnie) taki sam jak dla jednej oferty
offers["dwie_oferty"] = offers["liczba_ofert"] == 1
(offers.groupby("rok")["dwie_oferty"].mean() * 100).round(1)

## 3. Cena za Mg per rok: średnia, mediana, kwartyle, IQR

`df.groupby("rok")["cena_za_mg"]` = weź kolumnę z ceną, podziel na grupy wg roku i policz coś w każdej grupie. To dokładnie tabela przestawna z Excela. Puste ceny (`NaN`) pandas pomija sam.

In [ ]:
# groupby("rok") dzieli tabelę na grupy (jedna grupa = jeden rok);
# mean() liczy średnią ceny w każdej grupie: jedna liczba na rok
df.groupby("rok")["cena_za_mg"].mean().round(1)

**Twoja kolej:** linia z medianą jest gotowa; uruchom ją, potem podmień `median()` na wszystkie statystyki opisowe, a na koniec na jeden wybrany percentyl (cena, poniżej której jest np. 90% przetargów).

In [ ]:
# TODO: uruchom, potem podmień median() kolejno na describe() i quantile(0.9)
# Podpowiedź: describe() da 8 statystyk naraz (count, mean, std, min, 25%, 50%, 75%, max), osobno dla każdego roku;
# quantile(0.9) = cena, poniżej której jest 90% przetargów w danym roku
df.groupby("rok")["cena_za_mg"].median().round(1)

Mediana = środkowa wartość po ułożeniu cen od najniższej: połowa przetargów jest tańsza, połowa droższa. Kwartyl Q1 = cena, poniżej której jest 25% przetargów; Q3 = 75%.
IQR = Q3 - Q1, czyli "rozstęp środkowej połowy": im większy, tym bardziej rozrzucone ceny w danym roku.

In [ ]:
# agg([...]) = kilka statystyk naraz dla JEDNEJ kolumny;
# wynik to tabela z kolumnami count, mean, median
stats = df.groupby("rok")["cena_za_mg"].agg(["count", "mean", "median"])
# quantile(0.25) = Q1; dopisujemy jako nową kolumnę tabeli stats (pandas dopasuje wiersze po roku)
stats["q1"] = df.groupby("rok")["cena_za_mg"].quantile(0.25)
stats["q3"] = df.groupby("rok")["cena_za_mg"].quantile(0.75)
# IQR = kolumna minus kolumna, wiersz po wierszu
stats["iqr"] = stats["q3"] - stats["q1"]
stats = stats.round(1)
stats

Mediana rośnie z ok. 517 zł (2019) do ok. 923 zł (2024), czyli o prawie 80% w pięć lat; IQR rośnie z ok. 89 do ok. 129, więc rozrzut cen też się powiększa. `count` przypomina, ile przetargów stoi za każdą liczbą.

## 4. Trendy rok do roku

Ten sam `groupby`, ale dla kilku kolumn naraz: lista kolumn siedzi w zmiennej `trend_cols`, więc w nawiasie kwadratowym jest nazwa zmiennej; `df.groupby("rok")[trend_cols]` znaczy to samo co `df.groupby("rok")[["liczba_ofert", "okres_mies", ...]]`, gdzie lista kolumn jest wpisana wprost (stąd podwójne nawiasy). Osobno średnia i osobno mediana, żeby tabele były czytelne. `size()` liczy przetargi w roku, także te z pustymi polami.

In [ ]:
# lista kolumn, dla których liczymy trend
trend_cols = ["liczba_ofert", "okres_mies", "wartosc_pln", "wolumen_mg"]
# mean() dla kilku kolumn naraz: jedna tabela, wiersze = lata, kolumny = średnie
trend_mean = df.groupby("rok")[trend_cols].mean().round(1)
# size() = liczba przetargów w roku (także z pustymi polami); dopisujemy jako kolumnę tabeli
trend_mean["przetargi"] = df.groupby("rok").size()
trend_mean

In [ ]:
# median() dla tych samych kolumn: mniej wrażliwa na pojedyncze wielkie umowy niż średnia
trend_median = df.groupby("rok")[trend_cols].median().round(1)
trend_median

In [ ]:
# pct_change() = zmiana w procentach względem poprzedniego wiersza (roku);
# pierwszy rok nie ma poprzednika, stąd NaN
# razy 100, żeby czytać jak procenty:
# 30.8 = mediana wartości umowy wzrosła o 30,8% wobec poprzedniego roku
(trend_median["wartosc_pln"].pct_change() * 100).round(1)

Co widać: średnia liczba ofert spada (1,7 w 2019 do 1,4 w 2024), wartość umów rośnie, mediana wolumenu lekko spada, okres nie ma wyraźnego trendu.
Uwaga na skalę: przy kilkudziesięciu przetargach na rok jedna wielka umowa potrafi przesunąć średnią, dlatego zmianę rok do roku liczymy na medianie.

## 5. Korelacje

Korelacja to liczba od -1 do 1 mówiąca, czy dwie kolumny "idą w parze": plus = rosną razem, minus = jedna rośnie, gdy druga maleje, 0 = brak związku. Siłę oceniamy orientacyjnie po liczbie bez znaku: poniżej 0,1 brak związku, 0,1–0,3 słaby, 0,3–0,5 umiarkowany, powyżej 0,5 silny.
`corr()` liczy domyślnie korelację Pearsona (tę zwykłą).

In [ ]:
# lista kolumn liczbowych do korelacji
cols = ["liczba_ofert", "okres_mies", "wartosc_pln", "wolumen_mg", "cena_za_mg"]
# astype(float) = wszystko na zwykłe liczby z przecinkiem
# (liczba_ofert jest Int64, a corr() woli float)
# corr() = korelacja Pearsona każdej kolumny z każdą;
# na przekątnej zawsze 1.00 (kolumna sama ze sobą)
correlations = df[cols].astype(float).corr().round(2)
correlations

Spearman liczy korelację na kolejności wartości (miejscach w rankingu), nie na samych wartościach. Jest bezpieczniejszy dla "skokowej" liczby ofert (1, 2, 3) i odporny na wartości odstające (skrajne liczby typu 11 000 zł/Mg; w naszej tabeli takie ceny są już puste, w innych danych mogą zostać).

In [ ]:
# method="spearman" = korelacja rang;
# jeśli wynik wygląda podobnie jak Pearson, wniosek jest solidniejszy
df[cols].astype(float).corr(method="spearman").round(2)

`-0.00` w tabeli to zero po zaokrągleniu (minus został z bardzo małej ujemnej liczby). Do raportu wystarczy jedna kolumna tabeli Pearsona: jak liczba ofert koreluje z resztą.

In [ ]:
# interesuje nas jedna kolumna tabeli korelacji Pearsona: jak liczba ofert koreluje z resztą;
# sort_values() układa od najsilniejszej ujemnej
correlations["liczba_ofert"].sort_values()

Liczba ofert a cena za Mg: ok. -0,24, czyli słaba ujemna zależność (więcej ofert, niższa cena); z wartością i wolumenem słabe ujemne (-0,18 i -0,13), z okresem umowy brak (0,05).
Korelacja wartości z wolumenem (0,95) to oczywistość, bo wartość = cena razy wolumen; takich "odkryć" nie wpisuj do raportu. Korelacja to nie przyczynowość: mówi, że coś współwystępuje, nie dlaczego.

## 6. Eksport wyników do Excela

`pd.ExcelWriter` = jeden plik Excela z wieloma arkuszami. `with ... as writer:` otwiera plik i sam go zamyka po ostatniej wciętej linii (wcięcie działa jak w `if`).
Nazwy arkuszy: krótkie, bez polskich znaków, maks. 31 znaków (ograniczenie Excela).

In [ ]:
# with ... as writer: plik jest otwarty tylko w środku wcięcia;
# każdy to_excel(writer, sheet_name=...) to osobny arkusz
# Pułapka: jeśli ten plik jest otwarty w Excelu, zapis się nie uda (PermissionError);
# zamknij Excel i uruchom ponownie
with pd.ExcelWriter("wyniki/analiza_przetargi.xlsx") as writer:
    stats.to_excel(writer, sheet_name="cena_per_rok")
    offers_share.to_excel(writer, sheet_name="udzial_ofert")
    offers_by_year.to_excel(writer, sheet_name="udzial_ofert_rok")
    single_bid_by_year.to_excel(writer, sheet_name="jedna_oferta_rok")
    correlations.to_excel(writer, sheet_name="korelacje")
print("Zapisano: wyniki/analiza_przetargi.xlsx")

## 7. Wykres (opcjonalnie, jeśli zostanie czas)

matplotlib to biblioteka do wykresów; pandas ma do niej skrót `.plot()`. `savefig` zapisuje obrazek PNG do raportu, `show()` pokazuje go pod komórką. Bez stylizacji: cel to szybko zobaczyć trend.

In [ ]:
import matplotlib.pyplot as plt   # biblioteka do wykresów; "plt" to przyjęty skrót

# plot(kind="line") = wykres liniowy: oś X to rok (indeks tabeli, czyli etykiety wierszy), oś Y to mediana;
# marker="o" = kropka na każdym roku
stats["median"].plot(kind="line", marker="o", title="Mediana ceny za Mg (zł) per rok")
# savefig zapisuje obrazek do pliku; MUSI być przed show(), bo show() czyści wykres
plt.savefig("wyniki/mediana_ceny.png")
plt.show()

In [ ]:
# kind="bar" = słupki; single_bid_by_year to tabela z udziałem jednej oferty per rok w procentach
single_bid_by_year.plot(kind="bar", title="Udział przetargów z jedną ofertą (%)")
plt.savefig("wyniki/jedna_oferta.png")
plt.show()

## 8. Koncentracja wykonawców (opcjonalnie, jeśli zostanie czas)

Kto wygrywa najczęściej i jaki ma udział w wartości wszystkich umów (założenie 6). CR4 = łączny udział czterech największych firm; HHI = suma kwadratów udziałów w procentach, skala 0–10 000, im wyżej, tym bardziej skoncentrowany rynek.
Przetargi bez wykonawcy (12 pustych) pandas pomija sam.

In [ ]:
# value_counts() = ile przetargów wygrał każdy wykonawca; head(10) = 10 najczęstszych
df["wykonawca"].value_counts().head(10)

In [ ]:
# groupby("wykonawca") + sum() = łączna wartość wygranych umów per firma;
# sort_values od największej
value_by_winner = df.groupby("wykonawca")["wartosc_pln"].sum().sort_values(ascending=False)
# udział w % = wartość firmy / wartość wszystkich umów razy 100
market_share = (value_by_winner / value_by_winner.sum() * 100).round(2)
market_share.head(10)

In [ ]:
# CR4 = suma udziałów 4 największych;
# head(4) bierze 4 pierwsze wiersze (tabela jest już posortowana malejąco)
cr4 = market_share.head(4).sum()
# HHI = suma kwadratów udziałów w procentach; ** 2 = do kwadratu (** to potęga)
hhi = (market_share ** 2).sum()
print(f"CR4: {cr4:.1f}%   HHI: {hhi:.0f}")

Na tym zbiorze CR4 to ok. 43%, a HHI ok. 620, czyli rynek liczony "ogółem" wygląda na rozproszony. Zastrzeżenie z założenia 6: to cały kraj jako jeden rynek, a każda `MZK <Gmina>` liczy się osobno; przy rynkach lokalnych wynik może być zupełnie inny.

## Zadania

Jeśli na zajęciach brakuje czasu, zadania zostają na pracę domową. Rozwiązania są niżej, ale najpierw spróbuj sam.

1. Mediana ceny za Mg per województwo, od najdroższego województwa do najtańszego.
2. Średnia liczba ofert per tryb postępowania. W zbiorze są dwa tryby (`przetarg nieograniczony` i `tryb podstawowy`), więc tabela będzie mieć dwa wiersze.
3. Z tabeli `single_bid_by_year` (udział jednej oferty per rok, w procentach) odczytaj udział jednej oferty w 2024 i 2019 i wypisz różnicę w punktach procentowych.
4. Zapisz tabele trendów `trend_mean` i `trend_median` (średnie i mediany per rok) do nowego pliku `wyniki/trendy.xlsx`, każdą na osobnym arkuszu.
5. (bonus) Wykres liniowy średniej liczby ofert per rok, zapisany do `wyniki/srednia_ofert.png`.

In [ ]:
# Zadanie 1
# TODO: df.groupby("wojewodztwo")["cena_za_mg"].median(),
#       potem .round(1) i .sort_values(ascending=False).
# Podpowiedź: to samo co mediana ceny per rok, tylko w groupby stoi "wojewodztwo" zamiast "rok".

In [ ]:
# Zadanie 2
# TODO: groupby po kolumnie "tryb", wybierz "liczba_ofert", policz mean() i zaokrąglij do 2 miejsc.
# Podpowiedź: pandas sam pominie puste pola w liczbie ofert; wyjdą dwa wiersze, po jednym na tryb.

In [ ]:
# Zadanie 3
# TODO: single_bid_by_year[2024] to jedna liczba (udział w %);
#       odejmij single_bid_by_year[2019], zapisz do zmiennej change i wypisz przez print(f"...{change:.1f}...").
# Podpowiedź: f przed cudzysłowem = tekst, w który {nawiasy klamrowe} wstawiają wartość; :.1f = 1 miejsce po przecinku;
# rok w nawiasach kwadratowych bez cudzysłowu, bo etykiety wierszy (indeks) to liczby, nie teksty.

In [ ]:
# Zadanie 4
# TODO: with pd.ExcelWriter("wyniki/trendy.xlsx") as writer:
#       i w środku dwa to_excel(writer, sheet_name=...).
# Podpowiedź: nazwy arkuszy np. "srednia" i "mediana"; pamiętaj o wcięciu linii w środku with.

In [ ]:
# Zadanie 5 (bonus)
# TODO: df.groupby("rok")["liczba_ofert"].mean() zapisz do zmiennej,
#       potem na tej zmiennej .plot(kind="line", marker="o").
# Podpowiedź: na końcu plt.savefig("wyniki/srednia_ofert.png") i plt.show();
# jeśli sekcja 7 (Wykres) była pominięta, dodaj na górze import matplotlib.pyplot as plt.

## Rozwiązania

In [ ]:
# Rozwiązanie 1: groupby po województwie, mediana ceny, zaokrąglenie, sortowanie od najdroższego
df.groupby("wojewodztwo")["cena_za_mg"].median().round(1).sort_values(ascending=False)

In [ ]:
# Rozwiązanie 2: groupby po trybie i średnia liczby ofert: dwa wiersze, po jednym na tryb;
# w naszym zbiorze oba tryby mają po zaokrągleniu tę samą średnią (1,58 i 1,58), więc tryb nie różnicuje liczby ofert
df.groupby("tryb")["liczba_ofert"].mean().round(2)

In [ ]:
# Rozwiązanie 3: pojedyncza wartość z tabeli po etykiecie wiersza (indeksie): rok jako liczba, bez cudzysłowu
share_2019 = single_bid_by_year[2019]
share_2024 = single_bid_by_year[2024]
# różnica dwóch procentów to punkty procentowe, nie procenty
change = share_2024 - share_2019
print(f"2019: {share_2019:.1f}%, 2024: {share_2024:.1f}%, zmiana: {change:.1f} pkt proc.")

In [ ]:
# Rozwiązanie 4: nowy plik Excela, dwie tabele trendów na dwóch arkuszach
with pd.ExcelWriter("wyniki/trendy.xlsx") as writer:
    trend_mean.to_excel(writer, sheet_name="srednia")
    trend_median.to_excel(writer, sheet_name="mediana")
print("Zapisano: wyniki/trendy.xlsx")

In [ ]:
# Rozwiązanie 5 (bonus): średnia liczby ofert per rok jako wykres liniowy
import matplotlib.pyplot as plt   # import potrzebny, jeśli sekcja 7 (Wykres) była pominięta

# najpierw tabela (średnia per rok) do zmiennej, potem wykres z tej zmiennej
mean_offers_by_year = df.groupby("rok")["liczba_ofert"].mean()
mean_offers_by_year.plot(kind="line", marker="o", title="Średnia liczba ofert per rok")
plt.savefig("wyniki/srednia_ofert.png")   # zapis przed show()
plt.show()

**Podsumowanie**
- Nowa kolumna to działanie na kolumnach: `df["cena_za_mg"] = df["wartosc_pln"] / df["wolumen_mg"]`; zawsze sprawdź `describe()` i skrajne wiersze, zanim policzysz dalej.
- `groupby("rok")["kolumna"].mean()` / `.median()` / `.quantile()` to tabela przestawna; `crosstab` liczy udziały w tabeli rok x liczba ofert.
- `corr()` daje korelacje od -1 do 1; Spearman jest bezpieczniejszy dla liczby ofert; korelacja to nie przyczynowość.
- `pd.ExcelWriter` zapisuje wiele tabel do jednego pliku Excela; `.plot()` + `plt.savefig()` daje wykres do raportu. Sekcje 3 (Cena za Mg per rok: średnia, mediana, kwartyle, IQR), 4 (Trendy rok do roku), 5 (Korelacje) i 6 (Eksport wyników do Excela) przeczytaj jeszcze raz w domu. Części oznaczone "(opcjonalnie, jeśli zostanie czas)", czyli sekcje 7 (Wykres) i 8 (Koncentracja wykonawców), oraz zadania mogą zostać na pracę domową.

Dalej: notebook 08 (Praca domowa)